# Weather Gold

Executable SQL transformations and orchestration live in this notebook. Only runtime checks are imported from `src.validation.gold`. Execute weather → air quality → earthquakes. All persistent writes are restricted to `weather_api_test.gold`. No job or dashboard is created.

## Parameters
`project_root` must contain this checkout and the notebook JSON sources. The catalog/schema parameters are validated against the authorized test target.

In [0]:
# Canonical bootstrap: filesystem/config/definition loading only, no database access.
from pathlib import Path
import ast
import hashlib
import json
import sys

def resolve_project_root(value='',*,start=None):
    if value:
        path=Path(value).expanduser()
        if not path.is_absolute():raise ValueError('project_root override must be absolute; leave blank for discovery')
        candidates=[path.resolve()]
    else:
        here=Path(start or Path.cwd()).resolve();candidates=[here,*here.parents]
    for candidate in candidates:
        if (candidate/'src/config.py').is_file() and (candidate/'notebooks/gold/weather_gold.ipynb').is_file():
            for name in ('weather','air_quality','earthquakes'):definition_cells(candidate,name)
            return candidate
    raise FileNotFoundError('Cannot discover project checkout. Supply absolute project_root containing src and notebooks/gold/*.ipynb JSON files (Databricks Git-folder export required).')

def resolve_config_path(root,value):
    path=Path(value).expanduser();path=path if path.is_absolute() else Path(root)/path
    if not path.is_file():raise FileNotFoundError('Missing complete pipeline config: '+str(path))
    return str(path.resolve())

def definition_cells(root,name):
    if name not in ('weather','air_quality','earthquakes'):raise ValueError('Unexpected predecessor '+name)
    path=Path(root)/'notebooks/gold'/(name+'_gold.ipynb')
    try:nb=json.loads(path.read_text())
    except (OSError,ValueError) as exc:raise ValueError('Missing/malformed notebook JSON '+str(path)+'; sync/export checkout source') from exc
    cells=[]
    for cell in nb.get('cells',[]):
        tags=cell.get('metadata',{}).get('tags',[])
        if not isinstance(tags,list) or any(not isinstance(t,str) for t in tags):raise ValueError('Malformed notebook definition tags: '+str(path))
        if 'gold_definitions' in tags:
            if cell.get('cell_type')!='code' or tags.count('gold_definitions')!=1:raise ValueError('Invalid definition cell '+str(path))
            source=''.join(cell['source']);tree=ast.parse(source)
            if any(not isinstance(node,(ast.Import,ast.ImportFrom,ast.Assign,ast.AnnAssign,ast.FunctionDef,ast.ClassDef)) for node in tree.body):raise ValueError('Unexpected executable top-level statement in definitions '+str(path))
            cells.append(source)
    if len(cells)!={'weather':3,'air_quality':2,'earthquakes':2}[name]:raise ValueError('Missing/extra gold_definitions tags in '+str(path))
    definitions=[node.name for source in cells for node in ast.parse(source).body if isinstance(node,(ast.FunctionDef,ast.ClassDef))]
    if len(definitions)!=len(set(definitions)):raise ValueError('Duplicate definitions in '+str(path))
    return cells

def load_gold_definitions(root,names,namespace):
    loaded=namespace.setdefault('_gold_loaded',{})
    if len(names)!=len(set(names)):raise ValueError('Duplicate predecessor request')
    for name in names:
        cells=definition_cells(root,name);fingerprint=hashlib.sha256(''.join(cells).encode()).hexdigest()
        if name in loaded:raise ValueError('Predecessor already loaded: '+name)
        known={node.name for prior in loaded for source in definition_cells(root,prior) for node in ast.parse(source).body if isinstance(node,(ast.FunctionDef,ast.ClassDef))}
        incoming={node.name for source in cells for node in ast.parse(source).body if isinstance(node,(ast.FunctionDef,ast.ClassDef))}
        if known&incoming:raise ValueError('Unexpected cross-notebook definition collision')
        for source in cells:exec(compile(source,str(Path(root)/'notebooks/gold'/(name+'_gold.ipynb')),'exec'),namespace)
        loaded[name]=fingerprint


In [0]:
# Minimal source locator; all validation/loading rules live in Weather bootstrap.
from pathlib import Path
import json
import sys
dbutils.widgets.text('project_root','')
_override=dbutils.widgets.get('project_root')
if _override and not Path(_override).expanduser().is_absolute():raise ValueError('project_root must be absolute, or blank for discovery')
_candidates=[Path(_override).expanduser().resolve()] if _override else [Path.cwd().resolve(),*Path.cwd().resolve().parents]
_root=next((p for p in _candidates if (p/'src/config.py').is_file() and (p/'notebooks/gold/weather_gold.ipynb').is_file()),None)
if _root is None:raise FileNotFoundError('Provide absolute project_root with src and notebook JSON sources; Git-folder source export may be required')
_boot=json.loads((_root/'notebooks/gold/weather_gold.ipynb').read_text())
_boot_cells=[c for c in _boot['cells'] if c.get('cell_type')=='code' and c.get('metadata',{}).get('tags')==['gold_bootstrap']]
if len(_boot_cells)!=1:raise ValueError('Expected exactly one canonical gold_bootstrap cell')
exec(compile(''.join(_boot_cells[0]['source']),'weather_gold bootstrap','exec'),globals())
project_root=resolve_project_root(str(_root))
sys.path.insert(0,str(project_root))
dbutils.widgets.text('config_path','config/isolated.example.json')
dbutils.widgets.dropdown('validate_only','true',['true','false'])
config_path=resolve_config_path(project_root,dbutils.widgets.get('config_path'))
dbutils.widgets.text("catalog", "weather_api_test")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("nearby_radius_km", "250")
dbutils.widgets.text("alignment_seconds", "900")
dbutils.widgets.text("grid_degrees", "2")
dbutils.widgets.text("stale_seconds", "10800")
dbutils.widgets.text("max_runs", "100")

dbutils.widgets.dropdown('job_mode','auto',['auto','true','false'])


## Shared SQL/session/configuration utilities
No connection or write is performed by definitions.

In [0]:
# gold_definitions: shared SQL infrastructure and WEATHER-owned transformations
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import uuid
import re
from src.config import load_config
from src.contracts import output_types
from src.validation.common import require, ValidationResult
from src.validation import gold as checks

CONTRACT_VERSION='gold-notebooks-v2'
EARTH_RADIUS_KM=6371.0088

@dataclass(frozen=True)
class GoldConfig:
    catalog:str='weather_api_test'
    schema:str='gold'
    nearby_radius_km:float=250.0
    alignment_seconds:float=900.0
    grid_degrees:float=2.0
    stale_seconds:float=10800.0
    max_runs:int=100
    def validate(self):checks.validate_gold_config(asdict(self));return self
    def table(self,name):
        import re
        require(re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',name) is not None,'Invalid Gold identifier',stage='gold_config')
        return f'{self.catalog}.{self.schema}.{name}'
    def json(self):return json.dumps(asdict(self),sort_keys=True)

class SQLExecutor:
    """Existing Spark .sql().collect() or SQL Connector; no session construction."""
    def __init__(self,session,*,verbose=True):
        import time
        self.session=session;self.verbose=verbose;self.query_count=0;self.query_seconds=0.;self.started=time.monotonic()
    def metrics(self):
        import time
        return {'query_count':self.query_count,'query_seconds':round(self.query_seconds,3),'elapsed_seconds':round(time.monotonic()-self.started,3)}
    def query(self,statement):
        import time
        started=time.monotonic();self.query_count+=1;operation=statement.split(None,1)[0].upper();ok=False
        try:
            if hasattr(self.session,'sql'):
                result=[r.asDict(recursive=True) for r in self.session.sql(statement).collect()]
            else:
                with self.session.cursor() as cursor:
                    cursor.execute(statement)
                    result=[dict(zip([c[0] for c in cursor.description],r)) for r in cursor.fetchall()] if cursor.description else []
            ok=True;return result
        finally:
            elapsed=time.monotonic()-started;self.query_seconds+=elapsed
            if self.verbose:print(json.dumps({'at':datetime.now(timezone.utc).isoformat(),'operation':operation,'query_count':self.query_count,'elapsed_seconds':round(elapsed,3),'status':'completed' if ok else 'failed'}),flush=True)

# No second implementation; SparkExecutor is an explicit adapter spelling for the same SQL protocol.
SparkExecutor=SQLExecutor

def literal(value):return "'"+str(value).replace("'","''")+"'"

def latest(relation,keys,order='snapshot_at DESC,source_file_path DESC'):
    return f'SELECT * FROM {relation} QUALIFY row_number() OVER(PARTITION BY {",".join(keys)} ORDER BY {order})=1'

def measurement_base(relation,registry,source,kind):
    """Genuinely shared identity/time projection, not source-specific business SQL."""
    keys=['location_id']+(['forecast_for'] if kind=='hourly' else ['forecast_date'] if kind=='daily' else [])
    columns=','.join('s.`'+n+'`' for n in output_types(source,kind))
    valid='observed_at' if kind=='current' else 'forecast_for' if kind=='hourly' else None
    extra=['r.timezone AS location_timezone']
    if valid:
        extra += [f'cast(s.{valid} AS DATE) AS valid_date_utc',f'hour(s.{valid}) AS valid_hour_utc',
          f'cast(from_utc_timestamp(s.{valid},r.timezone) AS TIMESTAMP_NTZ) AS valid_local_display',
          f'cast(from_utc_timestamp(s.{valid},r.timezone) AS DATE) AS valid_date_local',
          f'hour(from_utc_timestamp(s.{valid},r.timezone)) AS valid_hour_local',
          f'timestampdiff(SECOND,s.{valid},from_utc_timestamp(s.{valid},r.timezone)) AS local_utc_offset_seconds',
          f'timestampdiff(MICROSECOND,s.snapshot_at,s.{valid})/1000000D AS retrieval_to_valid_seconds',
          f"CASE WHEN s.{valid}<s.snapshot_at THEN 'before_retrieval' WHEN s.{valid}=s.snapshot_at THEN 'at_retrieval' ELSE 'after_retrieval' END AS valid_vs_retrieval"]
    else:extra += ["'UTC' AS daily_calendar_timezone"]
    return f'SELECT {columns},'+','.join(extra)+f' FROM ({latest(relation,keys)}) s LEFT JOIN {registry} r ON s.location_id=r.location_id'



## Weather transformations
Latest retrieval per stable key; UTC keys, explicit local displays, daily UTC calendar. Unknown weather codes remain Unknown.

In [0]:
WEATHER_CODES={0:'Clear sky',1:'Mainly clear',2:'Partly cloudy',3:'Overcast',45:'Fog',48:'Depositing rime fog',51:'Light drizzle',53:'Moderate drizzle',55:'Dense drizzle',56:'Light freezing drizzle',57:'Dense freezing drizzle',61:'Slight rain',63:'Moderate rain',65:'Heavy rain',66:'Light freezing rain',67:'Heavy freezing rain',71:'Slight snowfall',73:'Moderate snowfall',75:'Heavy snowfall',77:'Snow grains',80:'Slight rain showers',81:'Moderate rain showers',82:'Violent rain showers',85:'Slight snow showers',86:'Heavy snow showers',95:'Thunderstorm',96:'Thunderstorm with slight hail',99:'Thunderstorm with heavy hail'}
def weather_case(column='weather_code'):
    return 'CASE '+column+' '+ ' '.join(f'WHEN {n} THEN {literal(label)}' for n,label in WEATHER_CODES.items())+" ELSE 'Unknown' END"

def radiation_energy_mj(power,interval_seconds):
    """W/m² × actual known seconds / 1e6 = MJ/m²; unknown interval stays null."""
    return f'({power}*{interval_seconds}/1000000D)'

def weather_query(relation,registry,kind):
    core='temperature_2m IS NULL' if kind!='daily' else '(temperature_2m_min IS NULL OR temperature_2m_max IS NULL)'
    base=measurement_base(relation,registry,'weather',kind)
    if kind=='hourly':
        # Source hourly means describe the preceding hour. Only infer a duration
        # where a consecutive hourly step is actually present; first/gap stays null.
        base=f"SELECT b.*,CASE WHEN timestampdiff(SECOND,lag(forecast_for) OVER(PARTITION BY location_id ORDER BY forecast_for),forecast_for)=3600 THEN 3600 ELSE cast(NULL AS INT) END AS known_interval_seconds FROM ({base}) b"
        energy=','+radiation_energy_mj('shortwave_radiation','known_interval_seconds')+' AS shortwave_energy_known_interval_mj_m2'
    else:energy=''
    return f'SELECT b.*,{weather_case()} AS weather_description,{core} AS missing_core_measurement{energy} FROM ({base}) b'

WEATHER_KEYS={'dim_location':['location_id'],'weather_current_latest':['location_id'],
 'weather_hourly_latest':['location_id','forecast_for'],'weather_daily_latest':['location_id','forecast_date']}

def weather_transformations(ctx):
    return {'dim_location':'SELECT * FROM '+ctx['registry'],**{
       'weather_'+kind+'_latest':weather_query(ctx['sources']['weather',kind],ctx['registry'],kind)
       for kind in ('current','hourly','daily')}}



## Eligibility, validation, release publication and reconciliation
Pinned Delta inputs and digest-verified committed cohorts. Whole-cohort completion is required. Versioned copies stay invisible until the final seismic notebook switches the shared pointer. A single writer is required.

In [0]:
# Shared release infrastructure is defined here, not hidden in a src Gold module.
# The other owning notebooks load only these definition cells, never execution cells.
SOURCE_TABLES={('weather','current'):'weather_current',('weather','hourly'):'weather_hourly',('weather','daily'):'weather_daily',
 ('air_quality','current'):'air_quality_current',('air_quality','hourly'):'air_quality_hourly',('earthquakes','history'):'earthquakes_event_history'}

def notebook_contract(root):
    content=[]
    for name in ('weather','air_quality','earthquakes'):
        nb=json.loads((Path(root)/'notebooks/gold'/f'{name}_gold.ipynb').read_text())
        content += definition_cells(root,name)
    content += [''.join(c['source']) for c in json.loads((Path(root)/'notebooks/gold/weather_gold.ipynb').read_text())['cells'] if c.get('metadata',{}).get('tags')==['gold_bootstrap']]
    content.append((Path(root)/'src/validation/gold.py').read_text())
    return hashlib.sha256((CONTRACT_VERSION+''.join(content)).encode()).hexdigest()

def _version(db,table):return int(db.query('DESCRIBE HISTORY '+table+' LIMIT 1')[0]['version'])
def _exists(db,cfg,name):
    return bool(db.query(f'SELECT table_name FROM {cfg.catalog}.information_schema.tables WHERE table_schema={literal(cfg.schema)} AND table_name={literal(name)}'))
def _managed(db,table):
    props=db.query('SHOW TBLPROPERTIES '+table)
    require(any(r['key']=='weather_pipeline.gold.contract' and r['value']==CONTRACT_VERSION for r in props),'Unmanaged Gold object collision: '+table,stage='gold_schema')
def _record(db,cfg,release_id):
    rows=db.query('SELECT * FROM '+cfg.table('_gold_releases')+' WHERE release_id='+literal(release_id))
    require(len(rows)==1,'Missing/duplicate Gold release receipt',stage='gold_publication')
    return rows[0]
def _active_id(db,cfg):
    rows=db.query('SELECT * FROM '+cfg.table('_gold_active'))
    require(len(rows)==1 and rows[0]['singleton']==1 and isinstance(rows[0]['release_id'],str),'Invalid complete active pointer singleton',stage='gold_controls')
    return rows[0]['release_id']

def activate_release(db,cfg,release_id,prior,verify,*,generation,writer_token,verify_prior):
    """Fenced pointer transaction only. Prior copies must be checked before rollback is allowed."""
    if prior==release_id:
        verify();return
    verify_prior()
    table=cfg.table('_gold_active')
    db.query(f"UPDATE {table} SET release_id={literal(release_id)},generation=generation+1 WHERE singleton=1 AND release_id={literal(prior)} AND generation={generation} AND writer_token={literal(writer_token)}")
    try:
        pointer=db.query('SELECT * FROM '+table)[0]
        require(pointer['release_id']==release_id and pointer['generation']==generation+1 and pointer['writer_token']==writer_token,'Stale candidate or lost writer fence',stage='gold_publication')
        verify()
    except Exception:
        # No unverified prior release is restored. Generation advances on rollback, preventing ABA.
        db.query(f"UPDATE {table} SET release_id={literal(prior)},generation=generation+1 WHERE singleton=1 AND release_id={literal(release_id)} AND generation={generation+1} AND writer_token={literal(writer_token)}")
        raise

class GoldContext(dict):
    def temp(self,name,query):
        relation=self['prefix']+name
        self['db'].query('CREATE OR REPLACE TEMP VIEW '+relation+' AS '+query)
        self['temps'].append(relation);return relation
    def close(self):
        try:
            for name in reversed(self['temps']):self['db'].query('DROP VIEW IF EXISTS '+name)
        finally:
            release_writer(self)
            if self.get('result') is not None:self['result']['metrics']=self['db'].metrics()

def controls(ctx):
    db=ctx['db'];cfg=ctx['cfg']
    for name,definition in (STATE_SCHEMAS.items() if not ctx.get('state_schema_verified') else []):
        require(_exists(db,cfg,name),'Gold v2 state must be provisioned/migrated explicitly; see README',stage='gold_controls')
        _managed(db,cfg.table(name))
        checks.compatible(db,cfg.table(name),{p.split()[0]:p.split()[1].lower() for p in definition.split(',')})
    ctx['state_schema_verified']=True
    pointer=db.query('SELECT * FROM '+cfg.table('_gold_active'));releases=db.query('SELECT * FROM '+cfg.table('_gold_releases'));stages=db.query('SELECT * FROM '+cfg.table('_gold_stages'))
    result=checks.validate_control_rows(pointer,releases,stages,contract=ctx['contract'])
    require(all(json.loads(r['settings'])['schema']==cfg.schema and json.loads(r['settings'])['catalog']==cfg.catalog for r in releases),'Receipt Gold destination differs from invocation',stage='gold_controls')
    require(all(r['source_config_id']==ctx['silver'].fingerprint for r in releases),'Receipt source configuration differs; reviewed migration required',stage='gold_controls')
    expected={ctx['silver'].registry,ctx['silver'].ledger,*[ctx['silver'].silver(*key) for key in SOURCE_TABLES],ctx['silver'].silver('earthquakes','latest')}
    require(all(set(json.loads(r['input_versions']))==expected for r in releases),'Receipt pinned table identities differ',stage='gold_controls')
    return result

def acquire_writer(ctx,stage):
    # A single conditional UPDATE is the lock acquisition, never check-then-insert.
    controls(ctx);token=uuid.uuid4().hex;db=ctx['db'];table=ctx['cfg'].table('_gold_active')
    db.query(f"UPDATE {table} SET writer_token={literal(token)},writer_stage={literal(stage)},writer_acquired_at=current_timestamp() WHERE singleton=1 AND writer_token IS NULL")
    pointer=db.query('SELECT * FROM '+table)[0]
    require(pointer['writer_token']==token,'Gold writer busy. No expiring lease: inspect the owner; never clear a live writer',stage='gold_writer')
    ctx['writer_token']=token
    controls(ctx)
    return pointer

def require_writer(ctx):
    rows=ctx['db'].query('SELECT * FROM '+ctx['cfg'].table('_gold_active'))
    return checks.validate_writer_pointer(rows,ctx.get('writer_token'))

def record_failure(ctx,exc):
    # Keep immutable data; record health independently from earlier successful validation.
    if not ctx.get('writer_token') or not ctx.get('release_id') or not ctx.get('failure_mutations_allowed'):return
    db=ctx['db'];cfg=ctx['cfg'];rid=ctx['release_id']
    pointer=db.query('SELECT * FROM '+cfg.table('_gold_active'))
    require(len(pointer)==1 and pointer[0]['writer_token']==ctx['writer_token'],'Cannot record failure without ownership',stage='gold_writer')
    active=pointer[0]['release_id']==rid
    db.query(f"UPDATE {cfg.table('_gold_releases')} SET health='UNHEALTHY'"+(",publication_state='FAILED'" if not active else '')+f" WHERE release_id={literal(rid)}")
    if ctx.get('suspect_release') and ctx['suspect_release']!=rid:
        db.query(f"UPDATE {cfg.table('_gold_releases')} SET health='UNHEALTHY' WHERE release_id={literal(ctx['suspect_release'])}")
    print(json.dumps({'stage':ctx.get('write_stage'),'release_id':rid,'health':'UNHEALTHY','active':active,'error_type':type(exc).__name__}))

def verify_release_copies(ctx,rid):
    if not rid:return
    db=ctx['db'];cfg=ctx['cfg']
    rows=db.query('SELECT * FROM '+cfg.table('_gold_stages')+' WHERE release_id='+literal(rid))
    checks.validate_publication_completion({r['stage_name'] for r in rows if r['status']=='VALIDATED'})
    for row in rows:
        for name,version in json.loads(row['output_versions']).items():
            table=cfg.table('_data_'+name);scope=' WHERE _release_id='+literal(rid)
            try:checks.reconcile(db,'(SELECT * FROM '+table+f' VERSION AS OF {version}'+scope+')','(SELECT * FROM '+table+scope+')')
            except checks.ValidationError:
                ctx['suspect_release']=rid
                raise

def finish_result(ctx,result):
    result['ready_for_next_stage']=not result['validate_only']
    result['metrics']=ctx['db'].metrics();ctx['result']=result
    return result

def prepare_context(session,config_path,gold=None,*,release_id=None,project_root='.',write_stage=None):
    project_root=resolve_project_root('' if str(project_root)=='.' else str(project_root))
    config_path=resolve_config_path(project_root,config_path)
    cfg=(gold or GoldConfig()).validate();silver=load_config(config_path)
    require(silver.catalog=='weather_api_test','Gold inputs must be weather_api_test',stage='gold_config')
    db=session if isinstance(session,SQLExecutor) else SQLExecutor(session);db.query("SET TIME ZONE 'UTC'")
    ctx=GoldContext(db=db,cfg=cfg,silver=silver,prefix='_g_'+uuid.uuid4().hex[:12]+'_',temps=[],stages={},write_stage=write_stage,release_id=release_id,output_versions={})
    try:
        contract=notebook_contract(project_root);ctx['contract']=contract
        if _exists(db,cfg,'_gold_active'):
            pointer=acquire_writer(ctx,write_stage) if write_stage else controls(ctx)
        else:
            require(not write_stage,'First write requires explicit v2 control provisioning; see migration_plan()',stage='gold_controls')
            pointer={'release_id':'','generation':0}
        predecessor=pointer['release_id'];generation=pointer['generation']
        tables=[silver.registry,silver.ledger]+[silver.silver(*key) for key in SOURCE_TABLES]+[silver.silver('earthquakes','latest')]
        saved=_record(db,cfg,release_id) if release_id else None
        if saved:
            checks.validate_candidate_fence(pointer,saved)
            require(saved['contract_id']==contract and saved['source_config_id']==silver.fingerprint,'Notebook/config contract differs from staged release',stage='gold_publication')
            cfg=GoldConfig(**json.loads(saved['settings'])).validate();ctx['cfg']=cfg
        versions=json.loads(saved['input_versions']) if saved else {t:_version(db,t) for t in tables}
        require(set(versions)==set(tables),'Incomplete pinned input receipt',stage='gold_eligibility')
        pinned={t:f'{t} VERSION AS OF {versions[t]}' for t in tables}
        ledger=db.query('SELECT * FROM '+pinned[silver.ledger]+f' LIMIT {cfg.max_runs*silver.max_manifest_files+1}')
        require(len(ledger)<=cfg.max_runs*silver.max_manifest_files,f'Gold ledger rows {len(ledger)} exceed configured bound {cfg.max_runs*silver.max_manifest_files}; increase max_runs deliberately; no truncation',stage='gold_eligibility')
        run_ids=sorted({r['run_id'] for r in ledger});require(len(run_ids)<=cfg.max_runs,f'Discovered {len(run_ids)} ledger-referenced runs; max_runs={cfg.max_runs}. Increase max_runs explicitly (1..1000); all retained runs are required, never latest-N',stage='gold_eligibility')
        manifests=[]
        for run in run_ids:
            rec=db.query(f"SELECT cast(content AS STRING) content FROM read_files({literal(str(silver.manifest_path(run)))},format=>'binaryFile')")
            require(len(rec)==1,'Missing exact committed manifest',stage='gold_eligibility');manifests.append(json.loads(rec[0]['content']))
        eligible=checks.eligible_manifests(manifests,ledger,silver);evidence=[]
        for m in eligible:
            for entry in m['files']:
                rows=db.query(f"SELECT cast(content AS STRING) content,sha2(content,256) digest FROM read_files({literal(entry['path'])},format=>'binaryFile')")
                require(len(rows)==1,'Missing committed raw file',stage='gold_eligibility')
                evidence.append(checks.raw_evidence(rows[0]['content'],rows[0]['digest'],entry,m))
        members=[]
        for e in evidence:
            x=e['entry'];identity='cast(NULL AS BIGINT)' if x['location_id'] is None else str(x['location_id'])
            members.append('('+','.join([literal(x['source']),literal(x['path']),identity,'cast('+literal(e['snapshot_at'])+' AS TIMESTAMP)'])+')')
        member_sql=('SELECT * FROM VALUES '+','.join(members)+' AS m(source,path,location_id,snapshot_at)' if members else
            'SELECT cast(NULL AS STRING) source,cast(NULL AS STRING) path,cast(NULL AS BIGINT) location_id,cast(NULL AS TIMESTAMP) snapshot_at WHERE false')
        member=ctx.temp('members',member_sql);registry=ctx.temp('registry','SELECT * FROM '+pinned[silver.registry]);sources={}
        for (source,kind),name in SOURCE_TABLES.items():
            identity='' if source=='earthquakes' else ' AND s.location_id=m.location_id'
            sources[source,kind]=ctx.temp('eligible_'+name,f'SELECT s.* FROM {pinned[silver.silver(source,kind)]} s JOIN {member} m ON s.source_file_path=m.path AND s.source=m.source AND s.snapshot_at=m.snapshot_at{identity}')
        checks.validate_sources(db,sources,registry,evidence).report()
        signature=json.dumps({'versions':versions,'manifests':eligible,'settings':asdict(cfg),'contract':contract},sort_keys=True)
        evidence_id=hashlib.sha256(signature.encode()).hexdigest()
        if saved:
            predecessor=saved['expected_predecessor'];generation=saved['expected_generation']
        elif pointer['release_id']:
            active_receipt=_record(db,cfg,pointer['release_id'])
            if active_receipt['evidence_id']==evidence_id:saved=active_receipt;predecessor=saved['expected_predecessor'];generation=saved['expected_generation']
        computed=hashlib.sha256((evidence_id+predecessor+str(generation)).encode()).hexdigest()[:32]
        require(release_id is None or computed==release_id,'Staged release evidence changed',stage='gold_eligibility')
        report={'eligible_runs':[m['run_id'] for m in eligible],'excluded_incomplete_runs':sorted(set(run_ids)-{m['run_id'] for m in eligible}),
                'discovered_runs':len(run_ids),'max_runs':cfg.max_runs,'ledger_rows':len(ledger),'eligible_files':len(evidence),'unit_fields_verified':sum(e['unit_fields_verified'] for e in evidence),
                'unit_defaults_used':sum(e['unit_defaults_used'] for e in evidence)}
        if saved is None and _exists(db,cfg,'_gold_releases'):
            existing=db.query('SELECT * FROM '+cfg.table('_gold_releases')+' WHERE release_id='+literal(computed))
            if existing:saved=existing[0]
        ctx.update(release_id=computed,expected_predecessor=predecessor,expected_generation=generation,evidence_id=evidence_id,versions=versions,contract=contract,evidence=evidence,registry=registry,sources=sources,
            eligible_paths=sorted(e['entry']['path'] for e in evidence),built_at=saved['built_at'] if saved else datetime.now(timezone.utc).isoformat(),report=report)
        ctx['failure_mutations_allowed']=True
        return ctx
    except Exception as exc:
        try:record_failure(ctx,exc)
        finally:ctx.close()
        raise

STATE_SCHEMAS={
 '_gold_active':'singleton INT,release_id STRING,generation BIGINT,writer_token STRING,writer_stage STRING,writer_acquired_at TIMESTAMP,deployment_contract STRING',
 '_gold_releases':'release_id STRING,contract_id STRING,source_config_id STRING,input_versions STRING,eligible_paths STRING,settings STRING,built_at TIMESTAMP,report STRING,published_at TIMESTAMP,expected_predecessor STRING,expected_generation BIGINT,evidence_id STRING,publication_state STRING,health STRING,verified_at TIMESTAMP',
 '_gold_stages':'release_id STRING,stage_name STRING,status STRING,counts_json STRING,completed_at TIMESTAMP,error_type STRING,output_versions STRING'}

def ensure_state(ctx):
    db=ctx['db'];cfg=ctx['cfg'];pointer=require_writer(ctx)
    previous=pointer['release_id']
    checks.validate_publication_state(_record(db,cfg,previous) if previous else None,ctx['versions'],ctx['eligible_paths'],ctx['contract'],reviewed_upgrade=True)
    rid=ctx['release_id']
    existing=db.query('SELECT * FROM '+cfg.table('_gold_releases')+' WHERE release_id='+literal(rid))
    if existing:checks.validate_candidate_fence(pointer,existing[0]);return
    require(pointer['release_id']==ctx['expected_predecessor'] and pointer['generation']==ctx['expected_generation'],'Pointer changed since candidate preparation',stage='gold_publication')
    values=[rid,ctx['contract'],ctx['silver'].fingerprint,json.dumps(ctx['versions'],sort_keys=True),json.dumps(ctx['eligible_paths']),cfg.json()]
    db.query(f'INSERT INTO {cfg.table("_gold_releases")} VALUES ('+','.join(literal(v) for v in values)+',cast('+literal(ctx['built_at'])+' AS TIMESTAMP),'+literal(json.dumps(ctx['report']))+',NULL,'+literal(ctx['expected_predecessor'])+','+str(ctx['expected_generation'])+','+literal(ctx['evidence_id'])+",'CANDIDATE','UNKNOWN',NULL)")

def completed_stages(ctx):
    receipt='(SELECT * FROM '+ctx['cfg'].table('_gold_stages')+' WHERE release_id='+literal(ctx['release_id'])+')'
    checks.check_relation(ctx['db'],receipt,['release_id','stage_name'],required=('status',),predicates={'status':"status NOT IN ('BUILDING','VALIDATED','FAILED')"})
    rows=ctx['db'].query('SELECT stage_name,status FROM '+ctx['cfg'].table('_gold_stages')+' WHERE release_id='+literal(ctx['release_id']))
    return {r['stage_name'] for r in rows if r['status']=='VALIDATED'}

def stage_status(ctx,stage,status,counts=None,error=None):
    cfg=ctx['cfg'];db=ctx['db'];pointer=require_writer(ctx)
    require(pointer['release_id']!=ctx['release_id'],'Never mutate validated active stage receipts',stage='gold_publication')
    db.query(f'''MERGE INTO {cfg.table('_gold_stages')} t USING (SELECT {literal(ctx['release_id'])} release_id,
      {literal(stage)} stage_name,{literal(status)} status,{literal(json.dumps(counts or {}))} counts_json,
      current_timestamp() completed_at,{literal(error) if error else 'cast(NULL AS STRING)'} error_type,
      {literal(json.dumps(ctx.get('output_versions',{})))} output_versions) s
      ON t.release_id=s.release_id AND t.stage_name=s.stage_name
      WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *''')

def persisted_dependencies(ctx,names):
    for name in names:
        ctx['stages'][name]=ctx.temp('dependency_'+name,'SELECT * FROM '+ctx['cfg'].table('_data_'+name)+' WHERE _release_id='+literal(ctx['release_id']))
        for view_name,query in copy_view_queries(ctx['cfg'],name).items():
            if checks.validate_view_definition(ctx['db'],ctx['cfg'],view_name,query,CONTRACT_VERSION):checks.compatible(ctx['db'],ctx['cfg'].table(view_name),checks.schema(ctx['db'],ctx['stages'][name]))

def stage_products(ctx,queries,keys):
    stages=ctx['stages'];counts={}
    for name,q in queries.items():
        stage=ctx.temp(name,f'SELECT {literal(ctx["release_id"])} _release_id,cast({literal(ctx["built_at"])} AS TIMESTAMP) _built_at,s.* FROM ({q}) s')
        counts[name]=checks.check_relation(ctx['db'],stage,keys[name]);stages[name]=stage
    return counts

def column_comment(name,product):
    common={'_release_id':'Validated Gold release identity; keep related queries on one release',
      '_built_at':'UTC release build start, separate from retrieval and publication',
      'snapshot_at':'UTC retrieval time, not necessarily forecast model issue time',
      'forecast_for':'Full UTC model valid timestamp, canonical hourly key',
      'forecast_date':'UTC source daily calendar, not local-day totals',
      'observed_at':'UTC source model valid time, not verified station observations',
      'ingested_at':'UTC Bronze ingestion time','source_file_path':'Committed raw-file lineage',
      'valid_local_display':'Local timestamp without timezone, never a standalone key',
      'local_utc_offset_seconds':'IANA location UTC offset at valid time, including DST',
      'retrieval_to_valid_seconds':'Signed valid time minus retrieval seconds, not model forecast lead',
      'depth_km':'USGS source depth in km, valid negative values preserved',
      'magnitude':'Logarithmic source magnitude, retain magnitude_type, never sum',
      'event_time':'UTC occurrence timestamp with epoch-millisecond precision',
      'latitude':'Event latitude degrees' if product.startswith(('seismic','earthquake')) else 'Requested registry latitude degrees, not returned API grid coordinate',
      'longitude':'Event longitude degrees' if product.startswith(('seismic','earthquake')) else 'Requested registry longitude degrees, not returned API grid coordinate',
      'elevation_m':'API response elevation in metres, distinct from registry elevation',
      'population':'Recorded unverified population, not used for weighted exposure',
      'shortwave_energy_known_interval_mj_m2':'MJ/m² over known preceding interval, first/gap stays null'}
    unit=(checks.WEATHER_UNITS|checks.AQ_UNITS).get(name)
    return common.get(name,('Source units: '+(unit or 'dimensionless')) if unit is not None else '')


def preflight_and_write(ctx,names,keys):
    db=ctx['db'];cfg=ctx['cfg'];rid=ctx['release_id'];pointer=require_writer(ctx);active=pointer['release_id']
    checks.validate_candidate_fence(pointer,_record(db,cfg,rid))
    schemas={name:checks.schema(db,ctx['stages'][name]) for name in names}
    # Preflight every stage output before writing any stage output.
    for name in names:
        target=cfg.table('_data_'+name);stage=ctx['stages'][name]
        for view_name,query in copy_view_queries(cfg,name).items():
            if checks.validate_view_definition(db,cfg,view_name,query,CONTRACT_VERSION):checks.compatible(db,cfg.table(view_name),schemas[name])
        if _exists(db,cfg,'_data_'+name):
            _managed(db,target);checks.compatible(db,target,checks.schema(db,stage))
        if active and active!=rid and (name.endswith('_latest') or name=='earthquake_snapshot_audit'):
            old=ctx.temp('previous_'+name,'SELECT * FROM '+target+' WHERE _release_id='+literal(active))
            version=('n.updated_at<o.updated_at OR (n.updated_at=o.updated_at AND n.snapshot_at<o.snapshot_at)' if name=='seismic_events_latest' else
                     'n.snapshot_at<o.snapshot_at' if name.endswith('_latest') else None)
            checks.no_regression(db,old,stage,keys[name],version)
        if _exists(db,cfg,name):
            metadata=db.query(f'SELECT table_type,comment FROM {cfg.catalog}.information_schema.tables WHERE table_schema={literal(cfg.schema)} AND table_name={literal(name)}')
            require(len(metadata)==1 and metadata[0]['table_type']=='VIEW' and (metadata[0]['comment'] or '').startswith('weather-pipeline Gold '+CONTRACT_VERSION),'Gold name collision: '+name,stage='gold_schema')
            checks.compatible(db,cfg.table(name),checks.schema(db,stage))
    for name in names:
        stage=ctx['stages'][name];target=cfg.table('_data_'+name)
        checks.validate_release_rows(db,stage,rid)
        require_writer(ctx)
        if active!=rid:
            db.query(f"CREATE TABLE IF NOT EXISTS {target} USING DELTA TBLPROPERTIES ('weather_pipeline.gold.contract'='{CONTRACT_VERSION}') AS SELECT * FROM {stage} WHERE false")
            existing_rows=db.query(f'SELECT count(*) n FROM {target} WHERE _release_id={literal(rid)}')[0]['n']
            if not existing_rows:db.query(f'INSERT INTO {target} SELECT * FROM {stage}')
        actual=ctx.temp('persisted_'+name,'SELECT * FROM '+target+' WHERE _release_id='+literal(rid))
        checks.reconcile(db,stage,actual)
        ctx['stages'][name]=actual
        ctx['output_versions'][name]=_version(db,target)
        for view_name,query in copy_view_queries(cfg,name).items():
            checks.validate_view_definition(db,cfg,view_name,query,CONTRACT_VERSION)
            columns=','.join('`'+n+'`'+(' COMMENT '+literal(column_comment(n,name)) if column_comment(n,name) else '') for n in schemas[name])
            db.query(f"CREATE VIEW IF NOT EXISTS {cfg.table(view_name)} ({columns}) COMMENT 'weather-pipeline Gold {CONTRACT_VERSION}' AS {query}")

def build_derived(ctx,queries,keys):
    result={};counts={}
    for name,q in queries.items():
        result[name]=ctx.temp('derived_'+name,q);counts[name]=checks.check_relation(ctx['db'],result[name],keys[name])
    checks.validate_derived(ctx['db'],ctx['stages'],result)
    checks.preflight_views(ctx['db'],ctx['cfg'],result,CONTRACT_VERSION)
    return result,counts

def candidate_mapping(ctx):
    # Queries in each owning notebook receive relations, not hidden business code.
    release=ctx.temp('settings',f"SELECT {literal(ctx['release_id'])} release_id,{literal(ctx['cfg'].json())} settings,cast({literal(ctx['built_at'])} AS TIMESTAMP) published_at")
    return {**ctx['stages'],'_gold_settings':release}

def public_mapping(ctx):
    cfg=ctx['cfg']
    return {name:cfg.table(name) for name in ctx['stages']}|{'_gold_settings':f"(SELECT r.* FROM {cfg.table('_gold_releases')} r JOIN {cfg.table('_gold_active')} p ON r.release_id=p.release_id)"}

def install_views(ctx,queries,keys):
    require_writer(ctx);db=ctx['db'];cfg=ctx['cfg']
    # Preflight the complete family before any DDL, including actual SQL text/owner.
    preflight_query_family(ctx,queries)
    for name,q in queries.items():
        db.query(f"CREATE VIEW IF NOT EXISTS {cfg.table(name)} COMMENT 'weather-pipeline Gold {CONTRACT_VERSION}' AS {q}")

def published_releases_query(cfg):
    return f'''SELECT r.* FROM {cfg.table('_gold_releases')} r JOIN
      (SELECT release_id FROM {cfg.table('_gold_stages')} GROUP BY release_id
       HAVING count(*)=3 AND count_if(status='VALIDATED' AND stage_name IN ('weather','air_quality','earthquakes'))=3 AND count(DISTINCT stage_name)=3) s
      ON r.release_id=s.release_id WHERE r.publication_state='PUBLISHED' AND r.health='HEALTHY' QUALIFY count(*) OVER(PARTITION BY r.release_id)=1 '''

def copy_view_queries(cfg,name):
    target=cfg.table('_data_'+name)
    history=f"SELECT d.* FROM {target} d JOIN ({published_releases_query(cfg)}) r ON d._release_id=r.release_id"
    active=f"SELECT d.* FROM {target} d JOIN (SELECT * FROM {cfg.table('_gold_active')} QUALIFY count(*) OVER()=1 AND singleton=1) p ON p.singleton=1 AND d._release_id=p.release_id JOIN {cfg.table('_gold_releases')} r ON d._release_id=r.release_id WHERE ((r.health='HEALTHY' AND r.publication_state IN ('READY','PUBLISHED')) OR (r.publication_state='LEGACY_ACTIVE' AND r.health='UNKNOWN'))"
    return {name:active,name+'_history':history}

def history_mapping(ctx):
    cfg=ctx['cfg']
    return {name:cfg.table(name+'_history') for name in ctx['stages']}|{'_gold_settings':'('+published_releases_query(cfg)+')'}

def install_history_views(ctx,queries,keys):
    install_views(ctx,{n+'_history':q for n,q in queries.items()},{n+'_history':k for n,k in keys.items()})

def publish_weather_task_value(dbutils,result,*,job_mode=None):
    if result['validate_only'] or not result.get('ready_for_next_stage'):return False
    if job_mode is None:
        # Same jobId/False-result convention as src.ingestion.databricks; no upstream changes.
        if not hasattr(dbutils,'notebook') or not hasattr(dbutils.notebook,'entry_point'):return False
        context=dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        job_mode=context.jobId().isDefined()
    if not job_mode:return False
    checks.validate_task_handoff(result,None)
    saved=dbutils.jobs.taskValues.set(key='release_id',value=result['release_id'])
    checks.validate_task_handoff(result,saved)
    return True


def run_weather(session,*,config_path,project_root='.',gold=None,validate_only=True):
    ctx=prepare_context(session,config_path,gold,project_root=project_root,write_stage=None if validate_only else 'weather')
    try:
        counts=stage_products(ctx,weather_transformations(ctx),WEATHER_KEYS)
        for kind in ('current','hourly','daily'):
            name='weather_'+kind+'_latest'
            checks.validate_latest(ctx['db'],ctx['sources']['weather',kind],ctx['stages'][name],WEATHER_KEYS[name])
        if not validate_only:
            ensure_state(ctx);active=_active_id(ctx['db'],ctx['cfg'])==ctx['release_id']
            if not active:stage_status(ctx,'weather','BUILDING')
            try:
                preflight_and_write(ctx,counts,WEATHER_KEYS)
                if not active:stage_status(ctx,'weather','VALIDATED',counts)
            except Exception as exc:
                if not active:stage_status(ctx,'weather','FAILED',error=type(exc).__name__)
                raise
        return finish_result(ctx,{'stage':'weather','release_id':ctx['release_id'],'validate_only':validate_only,'counts':counts,**ctx['report']})
    except Exception as exc:
        record_failure(ctx,exc);raise
    finally:ctx.close()


def _legacy_migration_plan(session=None,*,project_root='.',config_path='config/isolated.example.json',gold=None):
    """READ-ONLY plan generator. Returns SQL strings; NEVER executes DDL/DML.

    Load definitions from all three notebooks before calling. Apply only during a
    reviewed maintenance window, first in a disposable isolated Gold deployment.
    """
    root=resolve_project_root('' if str(project_root)=='.' else str(project_root))
    silver=load_config(resolve_config_path(root,config_path));cfg=(gold or GoldConfig()).validate()
    require(silver.catalog=='weather_api_test','Only test deployment can be planned',stage='gold_migration')
    contract=notebook_contract(root);statements=[];rollback=[];baseline={}
    if session is None:
        statements=[f'CREATE SCHEMA IF NOT EXISTS {cfg.catalog}.{cfg.schema}']
        statements += [f"CREATE TABLE {cfg.table(name)} ({definition}) USING DELTA TBLPROPERTIES ('weather_pipeline.gold.contract'='{CONTRACT_VERSION}')" for name,definition in STATE_SCHEMAS.items()]
        statements += [f"INSERT INTO {cfg.table('_gold_active')} VALUES (1,'',0,NULL,NULL,NULL,{literal(contract)})"]
        return {'mode':'FIRST_RUN_EMPTY_STATE','contract':contract,'sql':statements,'rollback':[],
                'preconditions':['All three controls absent; single administrator; no Gold workers/readers during provisioning']}
    db=session if isinstance(session,SQLExecutor) else SQLExecutor(session)
    tables=db.query(f"SELECT table_name,table_type,table_owner,comment FROM {cfg.catalog}.information_schema.tables WHERE table_schema={literal(cfg.schema)}")
    inventory={r['table_name']:r for r in tables}
    keys=WEATHER_KEYS|AQ_KEYS|SEISMIC_KEYS
    required=set(STATE_SCHEMAS)|{'_data_'+n for n in keys}|set(keys)|set(AQ_VIEW_KEYS)|set(SEISMIC_VIEW_KEYS)|{'publication_status'}
    require(set(inventory)==required,'Unexpected/missing legacy objects; review inventory before migration',stage='gold_migration')
    for name in set(STATE_SCHEMAS)|{'_data_'+n for n in keys}:
        props=db.query('SHOW TBLPROPERTIES '+cfg.table(name))
        require(any(r['key']=='weather_pipeline.gold.contract' and r['value']=='gold-notebooks-v1' for r in props),'Only v1-to-v2 migration is generated; later upgrades need a reviewed definition diff',stage='gold_migration')
        baseline[name]=_version(db,cfg.table(name))
    pointer=db.query('SELECT * FROM '+cfg.table('_gold_active'))
    releases=db.query('SELECT * FROM '+cfg.table('_gold_releases'));stages=db.query('SELECT * FROM '+cfg.table('_gold_stages'))
    checks.validate_legacy_upgrade(pointer,releases,stages,silver.fingerprint)
    active=pointer[0]['release_id']
    for name,business_keys in keys.items():
        target=cfg.table('_data_'+name);scope=' WHERE _release_id='+literal(active)
        checks.check_relation(db,'(SELECT * FROM '+target+scope+')',['_release_id']+business_keys)
        checks.reconcile(db,'(SELECT * FROM '+target+scope+')',cfg.table(name))
    owner=inventory['_gold_active']['table_owner']
    for name,r in inventory.items():
        require(r['table_owner']==owner,'Different object owners require explicit owner review',stage='gold_migration')
        if r['table_type']=='VIEW':
            require((r['comment'] or '').startswith('weather-pipeline Gold gold-notebooks-v1'),'Foreign legacy view',stage='gold_migration')
            ddl=next(iter(db.query('SHOW CREATE TABLE '+cfg.table(name))[0].values()))
            # SHOW CREATE may return a two-part identifier; qualify the destination explicitly.
            rollback.append(re.sub(r'^CREATE VIEW\s+[^\s(]+', 'CREATE OR REPLACE VIEW '+cfg.table(name), ddl, count=1))
    for name,definition in STATE_SCHEMAS.items():
        existing=checks.schema(db,cfg.table(name));missing=[part for part in definition.split(',') if part.split()[0] not in existing]
        require(bool(missing),'Already changed control schema; inspect partial migration',stage='gold_migration')
        statements.append(f"ALTER TABLE {cfg.table(name)} ADD COLUMNS ("+','.join(missing)+')')
    statements += [f"UPDATE {cfg.table('_gold_active')} SET generation=0,writer_token=NULL,writer_stage=NULL,writer_acquired_at=NULL,deployment_contract={literal(contract)} WHERE singleton=1"]
    for receipt in releases:
        rid=receipt['release_id'];failed=any(x['release_id']==rid and x['status']=='FAILED' for x in stages)
        state='LEGACY_ACTIVE' if rid==active else 'FAILED' if failed else 'CANDIDATE'
        health='UNKNOWN'
        statements.append(f"UPDATE {cfg.table('_gold_releases')} SET expected_predecessor='',expected_generation=0,evidence_id=sha2({literal('legacy:'+rid)},256),publication_state={literal(state)},health={literal(health)},verified_at="+'NULL'+' WHERE release_id='+literal(rid))
    for row in stages:
        names=checks.STAGE_PRODUCTS[row['stage_name']]
        versions={name:baseline['_data_'+name] for name in names} if row['status']=='VALIDATED' else {}
        statements.append(f"UPDATE {cfg.table('_gold_stages')} SET output_versions={literal(json.dumps(versions,sort_keys=True))} WHERE release_id={literal(row['release_id'])} AND stage_name={literal(row['stage_name'])}")
    for name in baseline:
        statements.append(f"ALTER TABLE {cfg.table(name)} SET TBLPROPERTIES ('weather_pipeline.gold.contract'='{CONTRACT_VERSION}')")
    query_map={}
    for name in keys:query_map.update(copy_view_queries(cfg,name))
    context={'cfg':cfg,'stages':dict.fromkeys(keys)}
    query_map.update(air_quality_views(public_mapping(context)));query_map.update(earthquake_views(public_mapping(context)))
    query_map.update({n+'_history':q for n,q in (air_quality_views(history_mapping(context))|earthquake_views(history_mapping(context))).items()})
    query_map.update(publication_status=publication_status_query(cfg),published_releases=published_releases_query(cfg))
    for name,query in query_map.items():
        # Preserve existing columns and unit comments on copy views.
        original=name.removesuffix('_history')
        columns=''
        if original in keys:
            fields=checks.schema(db,cfg.table('_data_'+original))
            columns=' ('+','.join('`'+n+'`'+(' COMMENT '+literal(column_comment(n,original)) if column_comment(n,original) else '') for n in fields)+')'
        statements.append(f"CREATE OR REPLACE VIEW {cfg.table(name)}{columns} COMMENT 'weather-pipeline Gold {CONTRACT_VERSION}' AS {query}")
    for name,version in baseline.items():require(_version(db,cfg.table(name))==version,'Gold changed while planning; discard plan',stage='gold_migration')
    require(notebook_contract(root)==contract,'Notebook sources changed while planning; discard plan',stage='gold_migration')
    return {'mode':'V1_TO_V2_REVIEW_ONLY','contract':contract,'active_release':active,'baseline_versions':baseline,
            'sql':statements,'legacy_view_rollback':rollback,'new_views':[n for n in query_map if n not in inventory],
            'preconditions':['Stop all Gold workers and dashboard reads; recheck baseline versions immediately before applying',
              'Reconfirm original source/manifest evidence; this plan checks copy integrity, not full raw-to-Silver fidelity',
              'Keep original receipts, data, IDs, timestamps and contract hashes; never execute this plan as a notebook import side effect'],
            'metrics':db.metrics()}


def preflight_query_family(ctx,queries):
    # Session-local reuse is safe only while this invocation holds the writer guard.
    checked=ctx.setdefault('checked_view_queries',{})
    for name,query in queries.items():
        if checked.get(name)==query:continue
        if checks.validate_view_definition(ctx['db'],ctx['cfg'],name,query,CONTRACT_VERSION):
            expected=ctx.temp('expected_view_schema_'+name,query)
            checks.compatible(ctx['db'],ctx['cfg'].table(name),checks.schema(ctx['db'],expected))
        checked[name]=query


def recover_writer(ctx,*,expected_token,confirmed_terminated=False,termination_evidence=''):
    """Administrative operation, not a lease/fence for arbitrary data-table writes.

    Caller must independently establish process/session termination. Token is opaque
    invocation ownership, not proof that SQL already submitted has stopped executing.
    """
    checks.validate_recovery_confirmation(confirmed_terminated=confirmed_terminated,termination_evidence=termination_evidence)
    cfg=ctx['cfg'];db=ctx['db'];pointer=controls(ctx)
    require(bool(expected_token) and pointer['writer_token']==expected_token,'Recovery owner changed; do not clear another writer',stage='gold_recovery')
    db.query(f"UPDATE {cfg.table('_gold_active')} SET writer_token=NULL,writer_stage=NULL,writer_acquired_at=NULL WHERE singleton=1 AND writer_token={literal(expected_token)} AND generation={pointer['generation']}")
    result=db.query('SELECT * FROM '+cfg.table('_gold_active'))[0]
    require(result['writer_token'] is None,'Recovery failed or another owner acquired the guard',stage='gold_recovery')
    return {'recovered_token':expected_token,'termination_evidence':termination_evidence,'generation':result['generation']}


MIGRATION_SCHEMA='singleton INT,migration_id STRING,plan_json STRING,checkpoint INT,expected_json STRING,inflight_index INT,status STRING,owner_token STRING'

def migration_snapshot(db,cfg,names):
    versions={name:_version(db,cfg.table(name)) for name in sorted(names)}
    objects=db.query(f"SELECT t.table_name,t.table_type,t.table_owner,t.comment,v.view_definition FROM {cfg.catalog}.information_schema.tables t LEFT JOIN {cfg.catalog}.information_schema.views v USING(table_catalog,table_schema,table_name) WHERE t.table_schema={literal(cfg.schema)} AND t.table_name<>'_gold_migration' ORDER BY t.table_name")
    columns=db.query(f"SELECT table_name,column_name,full_data_type,is_nullable,comment FROM {cfg.catalog}.information_schema.columns WHERE table_schema={literal(cfg.schema)} AND table_name<>'_gold_migration' ORDER BY table_name,ordinal_position")
    by_object={r['table_name']:hashlib.sha256(json.dumps([r,[x for x in columns if x['table_name']==r['table_name']]],sort_keys=True,default=str).encode()).hexdigest() for r in objects}
    return {'versions':versions,'metadata_hash':hashlib.sha256(json.dumps([objects,columns],sort_keys=True,default=str).encode()).hexdigest(),'objects':by_object}

def migration_json_sql(value):
    # Databricks regular SQL literals interpret backslash escapes. Encode bytes instead.
    payload=json.dumps(value,sort_keys=True,default=str).encode().hex()
    return "decode(unhex('"+payload+"'),'UTF-8')"

def migration_plan(session=None,*,project_root='.',config_path='config/isolated.example.json',gold=None):
    cfg=(gold or GoldConfig()).validate()
    if session is None:return _legacy_migration_plan(None,project_root=project_root,config_path=config_path,gold=cfg)
    db=session if isinstance(session,SQLExecutor) else SQLExecutor(session)
    root=resolve_project_root('' if str(project_root)=='.' else str(project_root))
    if _exists(db,cfg,'_gold_migration'):
        rows=db.query('SELECT * FROM '+cfg.table('_gold_migration'))
        require(len(rows)==1 and rows[0]['singleton']==1,'Invalid migration journal singleton',stage='gold_migration')
        plan=json.loads(rows[0]['plan_json'])
        require(plan['contract']==notebook_contract(root) and plan['target']==cfg.catalog+'.'+cfg.schema,'Journal/code/target differs; retain original reviewed code to resume',stage='gold_migration')
        checks.validate_migration_plan(plan,cfg)
        return plan
    plan=_legacy_migration_plan(db,project_root=root,config_path=config_path,gold=cfg)
    releases=db.query('SELECT * FROM '+cfg.table('_gold_releases'));stages=db.query('SELECT * FROM '+cfg.table('_gold_stages'))
    active=plan['active_release'];failed=sorted({r['release_id'] for r in stages if r['status']=='FAILED'})
    failed_sql=','.join(literal(x) for x in failed) or "''"
    release_sql=f"UPDATE {cfg.table('_gold_releases')} SET expected_predecessor='',expected_generation=0,evidence_id=sha2(concat('legacy:',release_id),256),publication_state=CASE WHEN release_id={literal(active)} THEN 'LEGACY_ACTIVE' WHEN release_id IN ({failed_sql}) THEN 'FAILED' ELSE 'CANDIDATE' END,health='UNKNOWN',verified_at=NULL"
    clauses=[]
    for stage,names in checks.STAGE_PRODUCTS.items():
        versions={name:plan['baseline_versions']['_data_'+name] for name in sorted(names)}
        clauses.append('WHEN '+literal(stage)+' THEN '+literal(json.dumps(versions,sort_keys=True)))
    stage_sql=f"UPDATE {cfg.table('_gold_stages')} SET output_versions=CASE WHEN status='VALIDATED' THEN CASE stage_name "+' '.join(clauses)+" ELSE '{}' END ELSE '{}' END"
    sql=[];replaced=set()
    for statement in plan['sql']:
        if statement.startswith('UPDATE '+cfg.table('_gold_releases')):
            if 'releases' not in replaced:sql.append(release_sql);replaced.add('releases')
        elif statement.startswith('UPDATE '+cfg.table('_gold_stages')):
            if 'stages' not in replaced:sql.append(stage_sql);replaced.add('stages')
        else:sql.append(statement)
    groups={'control_columns':[],'backfills':[],'metadata':[],'active_views':[],'historical_views':[]}
    for statement in sql:
        if ' ADD COLUMNS ' in statement:group='control_columns'
        elif statement.startswith('UPDATE '):group='backfills'
        elif statement.startswith('ALTER TABLE '):group='metadata'
        elif '_history' in statement.split(' COMMENT ',1)[0] or 'VIEW '+cfg.table('published_releases')+' ' in statement:group='historical_views'
        else:group='active_views'
        groups[group].append(statement)
    plan.update(sql=sql,groups=groups,target=cfg.catalog+'.'+cfg.schema,admission='LEGACY_ACTIVE_ONLY; new V2 build required for published history',
      invariant_checks=['Legacy controls validated before planning','Exact active copy/schema/key reconciliation before planning','Full V2 controls and all copy content checks before completion'],
      source_config_path=resolve_config_path(root,config_path),baseline=migration_snapshot(db,cfg,plan['baseline_versions']))
    plan['migration_id']=hashlib.sha256(json.dumps({'target':plan['target'],'contract':plan['contract'],'baseline':plan['baseline'],'sql':sql},sort_keys=True).encode()).hexdigest()
    return plan

def apply_migration_plan(session,plan,*,gold=None,stop_after=None,approved_existing_target=None):
    """Explicit maintenance executor; never called by normal stages/imports.

    Durable per-statement checkpoints support safe retry. An uncertain DDL/DML acknowledgement
    is fail-closed NEEDS_REVIEW, not permission to bypass version checks or blindly repeat.
    """
    cfg=(gold or GoldConfig()).validate();db=session if isinstance(session,SQLExecutor) else SQLExecutor(session)
    checks.validate_migration_plan(plan,cfg)
    require(plan['contract']==notebook_contract(resolve_project_root()),'Migration code differs from reviewed plan',stage='gold_migration')
    require(cfg.schema!='gold' or approved_existing_target=='weather_api_test.gold','Existing deployment requires separate explicit execution approval',stage='gold_migration')
    journal=cfg.table('_gold_migration')
    if not _exists(db,cfg,'_gold_migration'):
        require(migration_snapshot(db,cfg,plan['baseline_versions'])==plan['baseline'],'Unexpected initial baseline; discard plan',stage='gold_migration')
        db.query(f"CREATE TABLE {journal} ({MIGRATION_SCHEMA}) USING DELTA")
        db.query(f"INSERT INTO {journal} SELECT 1,{literal(plan['migration_id'])},{migration_json_sql(plan)},0,{migration_json_sql(plan['baseline'])},NULL,'PLANNED',NULL")
    rows=db.query('SELECT * FROM '+journal)
    require(len(rows)==1 and rows[0]['migration_id']==plan['migration_id'],'Migration journal identity mismatch',stage='gold_migration')
    row=rows[0]
    if row['status']=='COMPLETE':return {'status':'COMPLETE','checkpoint':row['checkpoint'],'already_complete':True}
    require(row['inflight_index'] is None,'Uncertain statement acknowledgement: NEEDS_REVIEW; inspect exact SQL and recorded Delta versions before administrator resolution',stage='gold_migration')
    require(migration_snapshot(db,cfg,plan['baseline_versions'])==json.loads(row['expected_json']),'Unexpected baseline changed since checkpoint; do not rebase automatically',stage='gold_migration')
    token=uuid.uuid4().hex
    db.query(f"UPDATE {journal} SET owner_token={literal(token)} WHERE singleton=1 AND owner_token IS NULL")
    require(db.query('SELECT owner_token FROM '+journal)[0]['owner_token']==token,'Another migration owner exists',stage='gold_migration')
    checkpoint=row['checkpoint'];expected=json.loads(row['expected_json'])
    try:
        for index in range(checkpoint,len(plan['sql'])):
            # Gate every later statement against every tracked Delta version and schema/view metadata.
            require(migration_snapshot(db,cfg,plan['baseline_versions'])==expected,'Unexpected baseline before statement '+str(index),stage='gold_migration')
            db.query(f"UPDATE {journal} SET inflight_index={index},status='APPLYING' WHERE singleton=1 AND owner_token={literal(token)}")
            db.query(plan['sql'][index])
            observed=migration_snapshot(db,cfg,plan['baseline_versions'])
            checks.validate_migration_transition(expected,observed,plan['sql'][index])
            expected=observed;checkpoint=index+1
            db.query(f"UPDATE {journal} SET checkpoint={checkpoint},expected_json={migration_json_sql(expected)},inflight_index=NULL,status='CHECKPOINTED' WHERE singleton=1 AND owner_token={literal(token)}")
            if stop_after is not None and checkpoint>=stop_after:return {'status':'CHECKPOINTED','checkpoint':checkpoint}
        ctx=GoldContext(db=db,cfg=cfg,silver=load_config(plan['source_config_path']),contract=plan['contract'],temps=[],prefix='_migration_',stages={})
        # Use explicit complete config matching the plan, never production fallback.
        ctx['silver']=load_config(plan['source_config_path'])
        controls(ctx);verify_release_copies(ctx,plan['active_release'])
        db.query(f"UPDATE {journal} SET status='COMPLETE' WHERE singleton=1 AND owner_token={literal(token)}")
        return {'status':'COMPLETE','checkpoint':checkpoint,'metrics':db.metrics()}
    except Exception:
        db.query(f"UPDATE {journal} SET status=CASE WHEN inflight_index IS NULL THEN 'BLOCKED' ELSE 'NEEDS_REVIEW' END WHERE singleton=1 AND owner_token={literal(token)}")
        raise
    finally:
        db.query(f"UPDATE {journal} SET owner_token=NULL WHERE singleton=1 AND owner_token={literal(token)}")


def release_writer(ctx):
    token=ctx.get('writer_token')
    if token:
        ctx['db'].query(f"UPDATE {ctx['cfg'].table('_gold_active')} SET writer_token=NULL,writer_stage=NULL,writer_acquired_at=NULL WHERE singleton=1 AND writer_token={literal(token)}")
        ctx['writer_token']=None


## Execute
Set `validate_only=false` deliberately to persist this stage. Pass the weather stage release_id unchanged to both later notebooks.

In [0]:
gold_result = run_weather(
    spark, config_path=config_path, project_root=project_root,
    gold=GoldConfig(
        catalog=dbutils.widgets.get("catalog"), schema=dbutils.widgets.get("gold_schema"),
        nearby_radius_km=float(dbutils.widgets.get("nearby_radius_km")),
        alignment_seconds=float(dbutils.widgets.get("alignment_seconds")),
        grid_degrees=float(dbutils.widgets.get("grid_degrees")),
        stale_seconds=float(dbutils.widgets.get("stale_seconds")),
        max_runs=int(dbutils.widgets.get("max_runs")),
    ), validate_only=dbutils.widgets.get("validate_only") == "true",
)


## Summary
A validated stage is not an active complete release until all three notebooks finish. Inspect `publication_status` after final publication.

In [0]:
gold_result['task_value_published']=publish_weather_task_value(dbutils,gold_result,job_mode={'auto':None,'true':True,'false':False}[dbutils.widgets.get('job_mode')])
print(json.dumps(gold_result, default=str, sort_keys=True))
